<a href="https://colab.research.google.com/github/AjayLohith/Naive-RAG/blob/main/Naive_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#BASIC RAG APP


In [61]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###Imports

In [62]:
!pip install openai chromadb python-dotenv

In [63]:
import os
import dotenv
from openai import OpenAI, api_key
from  dotenv import load_dotenv
import json
import chromadb
from openai.types.responses import responses_client_event
from websockets import client

In [64]:
from google.colab import userdata
groq_api_key=userdata.get('GROQ_API_KEY')

## Interacting with LLM

In [65]:
from openai import OpenAI

client=OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

In [66]:
response=client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role":"system","content":"You are a professional RAG expert in Agentic AI field"},
        {"role":"user","content":"Explainn me types of rags in 5 bullet points"}
    ],
    temperature=0
)
print(response.choices[0].message.content)

As a RAG (Retrieve, Augment, Generate) expert in Agentic AI, I'd be happy to break down the types of RAGs for you. Here are 5 key types:

* **Retrieve-only RAG**: This type of RAG focuses solely on retrieving relevant information from a knowledge base or database, without generating new text. It's often used for question-answering tasks or providing definitions.
* **Augment-only RAG**: In this type, the RAG augments existing text by adding, modifying, or removing content to improve its relevance, accuracy, or coherence. This is useful for tasks like text editing, summarization, or data enrichment.
* **Generate-only RAG**: This type of RAG generates entirely new text based on a given prompt, topic, or style. It's commonly used for creative writing, content generation, or chatbot responses.
* **Retrieve-and-Generate (RAG) RAG**: This type combines the retrieve and generate functions, where the RAG retrieves relevant information and then generates new text based on that information. This 

## Loading and chunking data from document

In [67]:
with open("/content/drive/MyDrive/Naive RAG/_data/company_hr_policy.txt","r")as f:
  hr_doc=f.read()

with open("/content/drive/MyDrive/Naive RAG/_data/engineering_standards.txt","r")as f:
  engineering_standards=f.read()

with open("/content/drive/MyDrive/Naive RAG/_data/onboarding_guide.txt","r")as f:
  onboarding_guide=f.read()

with open("/content/drive/MyDrive/Naive RAG/_data/product_knowledge_base.txt","r")as f:
  product_knowledge=f.read()

with open("/content/drive/MyDrive/Naive RAG/_data/security_policy.txt","r")as f:
  security_policy=f.read()

print(hr_doc)
# print(engineering_standards)
# print(onboarding_guide)
# print(product_knowledge)
# print(security_policy)


NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: January 2026

SECTION 1: LEAVE POLICY

Annual Leave:
All full-time employees are entitled to 24 days of paid annual leave per calendar year. Leave accrues at the rate of 2 days per month. New employees can start using accrued leave after completing 3 months of service. Unused leave up to 10 days can be carried forward to the next year. Any leave beyond 10 days will lapse on December 31st.

Sick Leave:
Employees are entitled to 12 days of sick leave per year. Sick leave for more than 3 consecutive days requires a medical certificate from a registered medical practitioner. Sick leave cannot be carried forward or encashed. In case of extended illness beyond 12 days, employees may apply for medical leave without pay, subject to HR approval.

Casual Leave:
Employees are entitled to 6 days of casual leave per year. Casual leave cannot be taken for more than 3 consecutive days. Prior approval from the reporting man

In [68]:
print(f"HR Document lenght: {len(hr_doc)} chars")
print(f"HR Document lenght: {len(engineering_standards)} chars")

print(f"HR document words {len(hr_doc.split())}")

HR Document lenght: 7464 chars
HR Document lenght: 4941 chars
HR document words 1052


## Chunking Strategy

In [69]:
def chunk_documents(text,source_name):
  paragraph=text.strip().split("\n" "\n")
  chunks=[]

  for para in paragraph:
    para=para.strip()
    if len(para)<50:
      continue

    if para.startswith("=="):
      continue

    chunks.append({"text":para,"source":source_name})


  return chunks

In [70]:
hr_chunk=chunk_documents(hr_doc,"HR Policy")
engineering_standards_chunk=chunk_documents(engineering_standards,"Engineering Standards")
onboarding_guide_chunk=chunk_documents(onboarding_guide,"Onboarding Guide")
product_knowledge_chunk=chunk_documents(product_knowledge,"Product Knowledge Base")
security_policy_chunk=chunk_documents(security_policy,"Security Policy")

In [71]:
total_chunks=hr_chunk+engineering_standards_chunk+onboarding_guide_chunk+product_knowledge_chunk+security_policy_chunk
print(f"Total Chunks: {len(total_chunks)}")

Total Chunks: 120


## Storing chunks into ChromDb

In [72]:
chroma_client=chromadb.Client()
collection=chroma_client.create_collection(name="company_chunks")

InternalError: Collection [company_chunks] already exists

In [73]:
documents=[]
ids=[]
metadata=[]

for i,chunk in enumerate(total_chunks):
  documents.append(chunk['text'])
  ids.append(f"chunk_{i}")
  metadata.append({"source": chunk["source"]})

print(documents[0])
print(ids[0])
print(metadata[0])

NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: January 2026
chunk_0
{'source': 'HR Policy'}


##### Here Chromadb automatically process embeddings using sentence-transformer

In [74]:
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadata
)

KeyboardInterrupt: 

In [75]:
print(f"Stored documents in ChromaDb :{len(documents)}")
# print(f"Stored documents in ChromaDb :{documents}")

Stored documents in ChromaDb :120


#### Behind the scenes of similarity seach

In [76]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

sentence1="I'm driving my Rolls-Royce Phantom on the highway at 70 mph."
sentence2="I'm eating grilled chicken with rice after finishing my workout."
sentence3="Im eating on chicken"

embed1=model.encode(sentence1)
embed2=model.encode(sentence2)
embed3=model.encode(sentence3)

print(embed1.shape)
print(embed2.shape)
print(embed3.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(384,)
(384,)
(384,)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity1=cosine_similarity([embed1],[embed2])
similarity2=cosine_similarity([embed2],[embed3])
similarity3=cosine_similarity([embed1],[embed3])

print(similarity1)
print(similarity2)
print(similarity3)


## Construction of Retrievel Pipeline

In [77]:
def retrieve(question,n_results=3):
  results=collection.query(
      query_texts=[question],
      n_results=n_results
  )
  return results['documents'][0],results['metadatas'][0]

In [78]:
chunks , sources = retrieve("What is the work from home policy?",5)

for i in range(len(chunks)):
  print(f"----Chunk {i+1}----")
  print(f"source:{sources[i]}")
  print(f"Texts: {chunks[i]}")
  print()

----Chunk 1----
source:{'source': 'HR Policy'}
Texts: Eligibility:
All employees who have completed their probation period (6 months) are eligible for Work From Home (WFH) arrangements. Employees in their probation period may request WFH only in exceptional circumstances with manager and HR approval.

----Chunk 2----
source:{'source': 'HR Policy'}
Texts: NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: January 2026

----Chunk 3----
source:{'source': 'HR Policy'}
Texts: Regular WFH:
Employees may work from home up to 2 days per week. The preferred WFH days are Wednesday and Friday, though teams may adjust based on project needs. Employees must be available during core working hours (10:00 AM to 6:00 PM IST) on WFH days.

----Chunk 4----
source:{'source': 'Security Policy'}
Texts: Internet Usage:
- Company internet is primarily for work purposes
- Limited personal use is acceptable during breaks
- The following are strictly prohibited: downloading copyrighte

# Testing RAG Pipeline

In [ ]:
ask_rag("What is the work from Home Policy?")

In [ ]:
ask_rag("How many days of annual leave do employees get?Just give me number no extra info")

In [79]:
def ask_rag(question,n_results=3,verbose=True):
  chunks,sources=retrieve(question,n_results)

  if verbose:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"\n{'-'*60}")
    print(f"Retrieved {len(chunks)} chunks:")

    for i,(chunks,sources) in enumerate(zip(chunks,sources)):
      print(f"[{sources['source']}] {chunks[:80]}...")
    print(f"\n{'-'*60}")

  context="\n \n".join(chunks)

  messages=[
      {
        "role":"system",
        "content":

        "You are a helpful assistance that asnwers quesitosn based ONLY on the provided context"
        "If the context does not contain informaiton to answer this quesitons, "
        "say I dont have enough context or information to answer this question"
        "Do not make up information or assume anything. Strictly answer only from the providede context"
    },
    {
        "role":"user",
        "content":f"Context: {context}\n \n---\n\nQuestion:{question}"
    }

  ]

  response=client.chat.completions.create(
      model=MODEL,
      messages=messages,
      temperature=0.2
  )
  answer=(response.choices[0].message.content)

  if verbose:
    print(f"Answer: {answer}")
    print(f"{'=' *60}")

  return answer
print("RAG pipeline is built")




RAG pipeline is built


In [80]:
ask_rag("What is the capital of Germany?")


Question: What is the capital of Germany?

------------------------------------------------------------
Retrieved 3 chunks:
[Product Knowledge Base] Data Storage:
All data is stored on AWS servers in the Mumbai (ap-south-1) regio...
[Product Knowledge Base] User Roles:
- Workspace Admin: Full control over billing, users, and settings. C...
[Product Knowledge Base] What is CloudDesk Pro?
CloudDesk Pro is a cloud-based project management and tea...

------------------------------------------------------------
Answer: I don't have enough context or information to answer this question. The provided context only talks about Cloud Desk Pro, a cloud-based project management platform, and does not mention Germany or its capital.


"I don't have enough context or information to answer this question. The provided context only talks about Cloud Desk Pro, a cloud-based project management platform, and does not mention Germany or its capital."

In [81]:
ask_rag("What happens during the probation period?")


Question: What happens during the probation period?

------------------------------------------------------------
Retrieved 3 chunks:
[HR Policy] Probation Period:
All new employees undergo a probation period of 6 months from ...
[HR Policy] Extension of Probation:
In cases where performance is borderline, probation may ...
[HR Policy] Eligibility:
All employees who have completed their probation period (6 months) ...

------------------------------------------------------------
Answer: During the probation period (6 months), employees who have completed their probation period are eligible for Work From Home (WFH) arrangements, except in exceptional circumstances with manager and HR approval.


'During the probation period (6 months), employees who have completed their probation period are eligible for Work From Home (WFH) arrangements, except in exceptional circumstances with manager and HR approval.'

In [86]:
# Product KB questions
ask_rag("What are the pricing plans for CloudDesk Pro?")


Question: What are the pricing plans for CloudDesk Pro?

------------------------------------------------------------
Retrieved 3 chunks:
[Product Knowledge Base] Support Channels:
- Help Center: docs.clouddesk.pro — Self-service articles and ...
[Product Knowledge Base] What is CloudDesk Pro?
CloudDesk Pro is a cloud-based project management and tea...
[Product Knowledge Base] CloudDesk Pro — Product Knowledge Base
Internal Support Reference | Version 2.1 ...

------------------------------------------------------------
Answer: I don't have enough context or information to answer this question. The provided context does not contain any information about pricing plans for CloudDesk Pro.


"I don't have enough context or information to answer this question. The provided context does not contain any information about pricing plans for CloudDesk Pro."

In [83]:
ask_rag("How do I cancel my subscription?")


Question: How do I cancel my subscription?

------------------------------------------------------------
Retrieved 3 chunks:
[Product Knowledge Base] "How do I cancel my subscription?":
Go to Settings → Billing → Subscription → Ca...
[Product Knowledge Base] Refund Policy:
Monthly subscriptions: Full refund if cancelled within 48 hours o...
[Product Knowledge Base] Free Trial:
All new accounts start with a 14-day free trial of the Business plan...

------------------------------------------------------------
Answer: I don't have enough context or information to answer this question. The provided context only mentions a free trial of a business plan, but it does not include any information about canceling a subscription.


"I don't have enough context or information to answer this question. The provided context only mentions a free trial of a business plan, but it does not include any information about canceling a subscription."

In [84]:
ask_rag("What is the refund policy?")


Question: What is the refund policy?

------------------------------------------------------------
Retrieved 3 chunks:
[Product Knowledge Base] Refund Policy:
Monthly subscriptions: Full refund if cancelled within 48 hours o...
[Product Knowledge Base] Failed Payments:
If a payment fails, the system retries 3 times over 7 days. If ...
[HR Policy] Reimbursement Process:
All expense claims must be submitted through the HR porta...

------------------------------------------------------------
Answer: All expense claims must be submitted through the HR portal within 30 days of incurring the expense. Claims submitted after 30 days may be rejected. Approved reimbursements are processed in the next payroll cycle. Original receipts or digital copies must be uploaded with each claim.


'All expense claims must be submitted through the HR portal within 30 days of incurring the expense. Claims submitted after 30 days may be rejected. Approved reimbursements are processed in the next payroll cycle. Original receipts or digital copies must be uploaded with each claim.'

In [85]:
ask_rag("Home Policy?")


Question: Home Policy?

------------------------------------------------------------
Retrieved 3 chunks:
[HR Policy] Eligibility:
All employees who have completed their probation period (6 months) ...
[Security Policy] NovaTech Solutions — Information Security Policy
Classification: Internal | Vers...
[HR Policy] Regular WFH:
Employees may work from home up to 2 days per week. The preferred W...

------------------------------------------------------------
Answer: I don't have enough context or information to answer this question. The provided context appears to be related to work schedules and employee availability, but it does not mention a "Home Policy".


'I don\'t have enough context or information to answer this question. The provided context appears to be related to work schedules and employee availability, but it does not mention a "Home Policy".'